Processo ETL GLUE - Tech Challenge Engenharia de Dados AWS

In [ ]:
import sys
import awswrangler as wr

from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions

from pyspark.context import SparkContext
from pyspark.sql import functions as F


args = getResolvedOptions(
    sys.argv,
    ["JOB_NAME"]
)

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init(args["JOB_NAME"], args)


query = """
WITH base_2022 AS (
    SELECT *, 2022 AS referencia
    FROM state_of_data_2022
),

base_2023 AS (
    SELECT *, 2023 AS referencia
    FROM state_of_data_2023
),

base_2024 AS (
    SELECT *, 2024 AS referencia
    FROM state_of_data_2024
),

base_2025 AS (
    SELECT *, 2025 AS referencia
    FROM state_of_data_2025
)

SELECT * FROM base_2022

UNION ALL

SELECT * FROM base_2023

UNION ALL

SELECT * FROM base_2024

UNION ALL

SELECT * FROM base_2025
"""

df_pandas = wr.athena.read_sql_query(
    sql=query,
    database="workspace_db",
    workgroup="primary"
)


df_spark = spark.createDataFrame(df_pandas)

# Remover registros completamente nulos
df_spark = df_spark.dropna(how="all")

# Remover duplicidades
df_spark = df_spark.dropDuplicates()

# Criar uma nova coluna
df_spark = df_spark.withColumn(
    "ano_processamento",
    F.year(F.current_date())
)

df_spark.printSchema()

df_spark.show(10, truncate=False)

(
    df_spark
    .write
    .mode("overwrite")
    .format("parquet")
    .save("s3://lab-378802791305/data_output/run_spec_tech_challenge/")
)


job.commit()